In [ ]:
## StringOutputParser is used to convert the output of the LLM into a plain string format. 
# This is important because sometimes the LLM returns structured objects, and we want to ensure that we are working with a simple string for further processing.

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0
)

template1 = PromptTemplate(template='Write a detailed report on {topic}' , input_variables=['topic'])
template2 = PromptTemplate(template='Write a 5 line summary on the following text \n {text}' , input_variables=['text'])

parser = StrOutputParser()
## Converts LLM output into a plain string
## Important because LLM returns structured objects sometimes

## This is a pipeline using LCEL (LangChain Expression Language) where the output of one component is passed as input to the next component.
chain = template1 | llm | parser | template2 | llm | parser
print(chain.invoke({'topic':'blackhole'})) 

c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Black holes are regions of spacetime with gravity so intense that nothing, not even light, can escape.
They form from the collapse of massive stars or through other processes, resulting in stellar-mass, intermediate-mass, and supermassive types.
Characterized by mass, spin, and charge, their presence is detected through gravitational influence on surroundings and emitted radiation.
Observations like gravitational waves and imaging the event horizon confirm Einstein's General Relativity.
Despite advancements, mysteries like the singularity and information paradox continue to drive scientific research.


In [ ]:
""" | Feature                              | `StrOutputParser` | `JSONOutputParser` | `StructuredOutputParser` | `PydanticOutputParser`       |
| ------------------------------------ | ----------------- | ------------------ | ------------------------ | ---------------------------- |
| Output Type                          | string            | dict               | dict                     | Pydantic object              |
| Structure Enforcement                | ❌ None            | ⚠️ Weak            | ✅ Strong                 | ✅ Very Strong                |
| Schema Definition                    | ❌                 | ❌                  | ✅ (ResponseSchema)       | ✅ (Pydantic Model)           |
| Type Validation                      | ❌                 | ❌                  | ❌ (only keys)            | ✅ (types + constraints)      |
| Field Descriptions                   | ❌                 | ❌                  | ✅                        | ✅                            |
| Error Handling                       | ❌                 | ❌                  | ❌                        | ⚠️ (raises validation error) |
| Reliability                          | Low               | Medium             | High                     | Very High                    |
| Ease of Use                          | Very Easy         | Easy               | Medium                   | Medium                       |
| Production Ready                     | ❌                 | ⚠️                 | ⚠️                       | ✅                            |
| Supports Constraints (e.g. age > 18) | ❌                 | ❌                  | ❌                        | ✅                            |
| Requires Format Instructions         | ❌                 | ✅                  | ✅                        | ✅                            |
 """

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv

# Parsers
from langchain_core.output_parsers import (
    JsonOutputParser,
    StrOutputParser,
    PydanticOutputParser
)
from langchain_classic.output_parsers import ResponseSchema , StructuredOutputParser


from pydantic import BaseModel, Field

load_dotenv()

# LLM
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0
)

# Same input for all
text = "This phone is amazing. The battery life is great and performance is smooth."

prompt_str = PromptTemplate(
    template="Summarize the following text:\n{text}",
    input_variables=["text"]
)

chain_str = prompt_str | model | StrOutputParser()

print("\n🔹 StrOutputParser Output:\n")
print(chain_str.invoke({"text": text}))


json_parser = JsonOutputParser()

prompt_json = PromptTemplate(
    template="""
Extract sentiment and key feature in JSON format.

{text}

{format_instructions}
""",
    input_variables=["text"],
    partial_variables={"format_instructions": json_parser.get_format_instructions()}
)

chain_json = prompt_json | model | json_parser

print("\n🔹 JsonOutputParser Output:\n")
print(chain_json.invoke({"text": text}))


schemas = [
    ResponseSchema(name="sentiment", description="positive or negative"),
    ResponseSchema(name="feature", description="main feature mentioned")
]

structured_parser = StructuredOutputParser.from_response_schemas(schemas)

prompt_struct = PromptTemplate(
    template="""
Extract the following information:

{text}

{format_instructions}
""",
    input_variables=["text"],
    partial_variables={"format_instructions": structured_parser.get_format_instructions()}
)

chain_struct = prompt_struct | model | structured_parser

print("\n🔹 StructuredOutputParser Output:\n")
print(chain_struct.invoke({"text": text}))


class Review(BaseModel):
    sentiment: str = Field(description="positive or negative")
    feature: str = Field(description="main feature mentioned")

pydantic_parser = PydanticOutputParser(pydantic_object=Review)

prompt_pydantic = PromptTemplate(
    template="""
Extract information from text.

{text}

{format_instructions}

ONLY return JSON.
""",
    input_variables=["text"],
    partial_variables={"format_instructions": pydantic_parser.get_format_instructions()}
)

chain_pydantic = prompt_pydantic | model | pydantic_parser

print("\n🔹 PydanticOutputParser Output:\n")
print(chain_pydantic.invoke({"text": text}))


🔹 StrOutputParser Output:

This phone has excellent battery life and performs smoothly.

🔹 JsonOutputParser Output:

{'sentiment': 'positive', 'key_features': ['battery life', 'performance']}

🔹 StructuredOutputParser Output:

{'sentiment': 'positive', 'feature': 'battery life'}

🔹 PydanticOutputParser Output:

sentiment='positive' feature='battery life'


In [2]:

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_core.output_parsers import JsonOutputParser

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0
)

# Create a JSON parser
# This will:
# 1. Tell LLM to return output in JSON format
# 2. Convert that JSON into Python dictionary
parser = JsonOutputParser()

template = PromptTemplate(template='Give me the name, age and city of a fictional person \n {format_instruction}',
                          input_variables=[], # No dynamic variables required from user input
                          # partial_variables automatically inject extra data into prompt
                          # Here we inject JSON formatting rules generated by parse
                          partial_variables={'format_instruction':parser.get_format_instructions()})

chain = template | llm | parser
result = chain.invoke({})
print(result)

{'name': 'Elara Vance', 'age': 29, 'city': 'Aethelburg'}


In [2]:

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_classic.output_parsers import ResponseSchema , StructuredOutputParser

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0
)

# Define schema (what structure we expect in output)
schema = [
    ResponseSchema(name='fact_1' , description='Fact 1 about topic'),
    ResponseSchema(name='fact_2' , description='Fact 2 about topic'),
    ResponseSchema(name='fact_3' , description='Fact 3 about topic'),
]

# Create parser from schema
# This will:
# 1. Generate format instructions for LLM
# 2. Parse output into structured dict
parser = StructuredOutputParser.from_response_schemas(schema)

template = PromptTemplate(template='Give 3 fact about {topic} \n {format_instruction}',
                          input_variables=['topic'],
                          partial_variables={'format_instruction':parser.get_format_instructions()})

prompt = template.invoke({'topic':'blackhole'})
result = llm.invoke(prompt)
final_result = parser.parse(result.content)
print(final_result)

{'fact_1': 'Black holes are regions in spacetime where gravity is so strong that nothing, not even light, can escape.', 'fact_2': 'The boundary of a black hole, beyond which escape is impossible, is called the event horizon.', 'fact_3': 'Supermassive black holes, millions or billions of times the mass of our Sun, are found at the centers of most galaxies, including our own Milky Way.'}


In [ ]:

from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_classic.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0
)

# Create parser using this schema
# It will:
# 1. Generate format instructions
# 2. Validate output against schema
# 3. Convert to Python object
class Person(BaseModel):
    name: str = Field(description='name of person')
    age: int = Field(description='age of person' , gt=18)
    city: str = Field(description='Name of city person belongs to')

parser = PydanticOutputParser(pydantic_object=Person)

template = PromptTemplate(template='Generate the name , age and city of a fictional {place} person \n {format_instruction}',
                          input_variables=['place'],
                          partial_variables={'format_instruction':parser.get_format_instructions()})

prompt = template.invoke({'place':'india'})
result = llm.invoke(prompt)
final_result = parser.parse(result.content)
print(final_result)

c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


name='Priya Sharma' age=28 city='Mumbai'
